In [3]:
import requests
from datetime import datetime
import logging
import json
import time

logging.basicConfig(level=logging.INFO)

API_ENDPOINT = "https://fake-json-api.mock.beeceptor.com/users"


In [4]:
def pipeline(url, output_file, params=None):
    logging.info("Starting pipeline")
    now = datetime.now()

    try:
        response = requests.get(url)
        response.raise_for_status()

        if response.status_code == 429:
            retry_time = int(response.headers.get("Retry-After"))
            logging.info(f"Rate limit exceeded, sleeping for {retry_time}")
            time.sleep(retry_time) 

        raw_data = response.json()

    except json.JSONDecodeError as e:
        logging.info(f"JSON Decode error: {e}")
    except Exception as e:
        logging.info(f"Exception occurred: {e}")

    transformed_data = []
    for user in raw_data:
        transformed_data.append({
            "id": user.get("id"),
            "name": user.get("name").strip().upper(),
            "username": user.get("username"),
            "email": user.get("email").strip().lower(),
            "zip": user.get("zip")[:5],
            "processed_at": datetime.strftime(now, "%Y-%m-%dT%H:%M:%S")
        })

    with open(output_file, mode="w", encoding="utf-8") as file:
        json.dump(transformed_data, file, indent=4)

    logging.info(f"Successfully processed {len(transformed_data)} records")
    

In [5]:
pipeline(API_ENDPOINT, "../output/transformed_users.json")

INFO:root:Starting pipeline
INFO:root:Successfully processed 10 records
